# Clusterização Hierárquica

Neste notebook, exploraremos os algoritmos de clusterização hierárquica, uma família de métodos que constrói uma hierarquia de clusters organizando os dados em uma estrutura semelhante a uma árvore. Diferentemente do K-Means, que requer que especifiquemos o número de clusters antecipadamente, a clusterização hierárquica nos permite descobrir a estrutura natural dos dados em diferentes níveis de granularidade.

A clusterização hierárquica pode ser dividida em duas abordagens principais:
- **Aglomerativa (Bottom-up)**: Inicia com cada ponto como um cluster individual e, iterativamente, combina os clusters mais próximos até formar um único cluster.
- **Divisiva (Top-down)**: Inicia com todos os pontos em um único cluster e, recursivamente, divide os clusters até que cada ponto forme seu próprio cluster.

Na prática quase só se usa a aglomerativa, porque a divisiva teria que escolher, a cada passo, entre $2^{|C|-1}-1$ maneiras de partir um cluster em dois. É ela que veremos aqui.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Fundamentação Matemática

A clusterização hierárquica aglomerativa funciona com base em uma **matriz de distâncias** entre todos os pares de pontos (ou clusters). O algoritmo segue estes passos fundamentais:

1. **Inicialização**: Cada observação $\mathbf{x}_i$ forma um cluster individual $C_i = \{\mathbf{x}_i\}$.

2. **Cálculo da Matriz de Distâncias**: Para $N$ pontos, calculamos uma matriz simétrica $D \in \mathbb{R}^{N \times N}$ onde $D_{ij}$ representa a distância entre os pontos $\mathbf{x}_i$ e $\mathbf{x}_j$:
   $$ D_{ij} = d(\mathbf{x}_i, \mathbf{x}_j) $$

3. **Iteração**: Em cada passo, encontramos o par de clusters $(C_i, C_j)$ com menor distância e os combinamos em um novo cluster $C_{ij} = C_i \cup C_j$.

4. **Atualização**: Recalculamos as distâncias do novo cluster para todos os outros clusters existentes.

5. **Terminação**: O processo continua até que reste apenas um cluster contendo todas as observações.

### Critérios de Ligação (Linkage)

O ponto crucial da clusterização hierárquica é como definimos a distância entre dois clusters. Existem vários critérios de ligação:

1. **Single Linkage (Ligação Simples)**:
   $$ d(C_i, C_j) = \min_{\mathbf{x} \in C_i, \mathbf{y} \in C_j} d(\mathbf{x}, \mathbf{y}) $$
   A distância é definida pelos pontos mais próximos entre os clusters.

2. **Complete Linkage (Ligação Completa)**:
   $$ d(C_i, C_j) = \max_{\mathbf{x} \in C_i, \mathbf{y} \in C_j} d(\mathbf{x}, \mathbf{y}) $$
   A distância é definida pelos pontos mais distantes entre os clusters.

3. **Average Linkage (Ligação Média)**:
   $$ d(C_i, C_j) = \frac{1}{|C_i||C_j|} \sum_{\mathbf{x} \in C_i} \sum_{\mathbf{y} \in C_j} d(\mathbf{x}, \mathbf{y}) $$
   A distância é a média de todas as distâncias entre pares de pontos dos clusters.

4. **Ward Linkage (Critério de Ward)**:
   $$\Delta(C_i, C_j) = \frac{|C_i||C_j|}{|C_i|+|C_j|} \|\mathbf{m}_i - \mathbf{m}_j\|^2$$
   Onde $\mathbf{m}_i$ e $\mathbf{m}_j$ são os centróides dos clusters $C_i$ e $C_j$, respectivamente, e $|C_k|$ é o número de pontos no cluster $C_k$.
   Minimiza a variância intra-cluster ao combinar clusters. É baseado na soma dos quadrados das distâncias aos centróides.

Dois detalhes sobre Ward: ele exige distância euclidiana, porque usa centróides; e a altura desenhada pelo SciPy não é $\Delta$, e sim $\sqrt{2\Delta}$. Para $P_0$ e $P_1$ do exemplo a seguir, $\Delta = 0{,}145$ mas o dendrograma mostra $0{,}539$.

## Implementação

Vamos construir uma implementação simplificada do algoritmo aglomerativo para entender seus passos fundamentais. Começamos com um exemplo de seis pontos, pequeno o bastante para acompanhar cada fusão na mão.

In [ ]:
X_simple = np.array([[1, 2], [1.5, 1.8], [5, 8], [8, 8], [1, 0.6], [9, 11]])

plt.figure(figsize=(8, 6))
plt.scatter(X_simple[:, 0], X_simple[:, 1], c='blue', s=100, alpha=0.7)
for i, (x, y) in enumerate(X_simple):
    plt.annotate(f'P{i}', (x, y), xytext=(5, 5), textcoords='offset points')
plt.title('Dataset Simples para Demonstração')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

In [ ]:
class HierarchicalClustering:
    def __init__(self, linkage='single'):
        self.linkage = linkage
        self.merge_history = []
        self.distances = []
        self.partitions = []

    def _distance_matrix(self, X):
        """Distância euclidiana entre todos os pares de pontos."""
        n = len(X)
        D = np.zeros((n, n))

        for i in range(n):
            for j in range(i + 1, n):
                D[i, j] = np.linalg.norm(X[i] - X[j])
                D[j, i] = D[i, j]

        return D

    def _cluster_distance(self, c1, c2, D):
        """Distância entre dois clusters, segundo o critério de ligação."""
        pairwise = []
        for i in c1:
            for j in c2:
                pairwise.append(D[i, j])

        if self.linkage == 'single':
            return min(pairwise)
        elif self.linkage == 'complete':
            return max(pairwise)
        # elif self.linkage == 'average':
        #     ...

    def fit(self, X):
        """Funde o par de clusters mais próximo até restar um só."""
        D = self._distance_matrix(X)
        clusters = [[i] for i in range(len(X))]

        self.merge_history = []
        self.distances = []
        self.partitions = [list(clusters)]

        while len(clusters) > 1:
            # Procura, entre todos os pares de clusters, o de menor distância
            best_distance = float('inf')
            best_i, best_j = 0, 1

            for i in range(len(clusters)):
                for j in range(i + 1, len(clusters)):
                    distance = self._cluster_distance(clusters[i], clusters[j], D)
                    if distance < best_distance:
                        best_distance = distance
                        best_i = i
                        best_j = j

            self.merge_history.append((clusters[best_i], clusters[best_j]))
            self.distances.append(best_distance)

            # Junta os dois clusters escolhidos. Removemos best_j primeiro
            # porque ele é o maior índice, e tirar best_i antes deslocaria ele.
            merged = clusters[best_i] + clusters[best_j]
            clusters.pop(best_j)
            clusters.pop(best_i)
            clusters.append(merged)

            self.partitions.append(list(clusters))

        return self

In [ ]:
hc = HierarchicalClustering(linkage='single').fit(X_simple)

print(f"Início:   {hc.partitions[0]}")

for step in range(len(hc.distances)):
    c1, c2 = hc.merge_history[step]
    print(f"\nPasso {step + 1}:  une {c1} e {c2}, a distância {hc.distances[step]:.3f}")
    print(f"          {hc.partitions[step + 1]}")

## Dendrogramas

Um **dendrograma** é a representação gráfica da hierarquia de clusters. É uma estrutura em forma de árvore que mostra a ordem e as distâncias nas quais os clusters foram combinados.

- **Eixo horizontal**: Representa as observações ou clusters.
- **Eixo vertical**: Representa a distância na qual os clusters foram unidos.
- **Altura dos ramos**: Indica a dissimilaridade entre os clusters combinados.

A partir daqui usamos a implementação do SciPy, que é otimizada e desenha o dendrograma. O `linkage` calcula a hierarquia inteira e devolve uma matriz $(n-1) \times 4$, com os dois clusters combinados, a distância da fusão e o tamanho do cluster resultante. No scikit-learn o mesmo algoritmo é o `AgglomerativeClustering(n_clusters=3, linkage='ward')`, que devolve direto os rótulos, sem a hierarquia.

In [ ]:
linkage_methods = ['single', 'complete', 'average', 'ward']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, method in enumerate(linkage_methods):
    # Calcular a matriz de ligação
    linkage_matrix = linkage(X_simple, method=method)
    
    # Criar o dendrograma
    dendrogram(linkage_matrix, ax=axes[i], labels=[f'P{j}' for j in range(len(X_simple))])
    axes[i].set_title(f'Dendrograma - {method.capitalize()} Linkage')
    axes[i].set_xlabel('Pontos de Dados')
    axes[i].set_ylabel('Distância')

plt.tight_layout()
plt.show()

Cada critério de ligação produz diferentes estruturas de cluster:

- **Single Linkage**: Tende a criar clusters elongados e pode sofrer do "efeito corrente" (chaining effect).
- **Complete Linkage**: Produz clusters mais compactos e esféricos.
- **Average Linkage**: Um meio-termo entre single e complete.
- **Ward Linkage**: Minimiza a variância intra-cluster, similar ao objetivo do K-Means.

## Determinando o Número de Clusters

Uma das grandes vantagens da clusterização hierárquica é que podemos "cortar" o dendrograma em diferentes alturas para obter diferentes números de clusters. Isso é feito traçando uma linha horizontal através do dendrograma.

In [ ]:
# Usar Ward linkage para o exemplo
linkage_matrix = linkage(X_simple, method='ward')

# Definir diferentes alturas de corte
cut_heights = [2.0, 4.0, 6.0]
colors = ['red', 'green', 'blue']

# Visualizar o dendrograma com diferentes cortes
plt.figure(figsize=(10, 6))
dendrogram(linkage_matrix, labels=[f'P{j}' for j in range(len(X_simple))])

for height, color in zip(cut_heights, colors):
    plt.axhline(y=height, color=color, linestyle='--', label=f'Corte em {height}')

plt.title('Dendrograma com Linhas de Corte')
plt.xlabel('Pontos de Dados')
plt.ylabel('Distância')
plt.legend()
plt.show()

In [ ]:
# fcluster corta a hierarquia: criterion='distance' corta por altura,
# criterion='maxclust' devolve um número fixo de clusters
print("Predição de clusters para diferentes alturas de corte:")
print("=" * 55)

for height in cut_heights:
    clusters = fcluster(linkage_matrix, height, criterion='distance')

    print()
    print(f"Altura de corte: {height}")
    print(f"Número de clusters: {len(np.unique(clusters))}")

    for cluster_id in np.unique(clusters):
        points = [f"P{j}" for j in range(len(X_simple)) if clusters[j] == cluster_id]
        print(f"  Cluster {cluster_id}: {points}")

In [ ]:
# Visualização dos clusters resultantes
fig, axes = plt.subplots(1, len(cut_heights), figsize=(15, 4))

for i, height in enumerate(cut_heights):
    clusters = fcluster(linkage_matrix, height, criterion='distance')
    axes[i].scatter(X_simple[:, 0], X_simple[:, 1], c=clusters, s=100, alpha=0.7, cmap='viridis')
    
    # Adicionar rótulos dos pontos
    for j, (x, y) in enumerate(X_simple):
        axes[i].annotate(f'P{j}', (x, y), xytext=(5, 5), textcoords='offset points')
    
    axes[i].set_title(f'Clusters (corte = {height})\n{len(np.unique(clusters))} clusters')
    axes[i].set_xlabel('Feature 1')
    axes[i].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## Aplicação ao Dataset Iris

Agora vamos aplicar a clusterização hierárquica ao dataset Iris e comparar os quatro critérios de ligação em dados reais.

In [ ]:
iris = load_iris()
# padroniza para o comprimento da pétala, que varia mais, não dominar a distância
X_iris = StandardScaler().fit_transform(iris.data[:, 2:])
y_iris = iris.target

methods = ['ward', 'complete', 'average', 'single']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, method in enumerate(methods):
    linkage_matrix = linkage(X_iris, method=method)

    # com 150 folhas, mostramos só os 20 últimos agrupamentos
    dendrogram(linkage_matrix, ax=axes[0, i], truncate_mode='lastp', p=20, no_labels=True)
    axes[0, i].set_title(f'Dendrograma - {method.capitalize()}')

    clusters = fcluster(linkage_matrix, 3, criterion='maxclust')

    axes[1, i].scatter(X_iris[:, 0], X_iris[:, 1], c=clusters, s=50, alpha=0.7, cmap='viridis')
    axes[1, i].set_title(f'Clusters - {method.capitalize()}')
    axes[1, i].set_xlabel('Comprimento da Pétala')
    axes[1, i].set_ylabel('Largura da Pétala')

plt.tight_layout()
plt.show()

In [ ]:
print(f"{'critério':10} tamanhos dos 3 clusters")
for method in methods:
    clusters = fcluster(linkage(X_iris, method=method), 3, criterion='maxclust')
    print(f"{method:10} {np.bincount(clusters)[1:]}")

print(f"\n{'real':10} {np.bincount(y_iris)}")

Average chega quase aos 50 por espécie. Ward e complete erram a fronteira entre versicolor e virginica, que se sobrepõem nas pétalas. E o single devolve `[50 99 1]`: uma espécie isolada, as outras duas fundidas e um ponto sozinho.

É o **efeito corrente**. Como basta um par de pontos próximos para unir dois clusters, uma trilha de pontos entre versicolor e virginica funde os dois grupos inteiros. Isso não faz do single um método ruim: ele é o único dos quatro capaz de achar clusters alongados ou em anel, justamente por seguir trilhas. Veremos essa ideia de novo no DBSCAN.

## Custo e Limitações

- **Memória $O(N^2)$**: a matriz de distâncias tem $N^2$ entradas. Em `float64`, 10 mil pontos são ~800 MB, e 100 mil seriam 80 GB.
- **Tempo**: a implementação ingênua acima é $O(N^3)$, porque recalcula todas as distâncias entre clusters a cada fusão. O SciPy chega a $O(N^2)$.
- **As fusões são definitivas**: o algoritmo é guloso e nunca desfaz uma união, ao contrário do K-Means, que reatribui pontos a cada iteração.
- **O dendrograma sempre existe**, inclusive sobre dados sem estrutura nenhuma. Se todas as fusões acontecem a alturas parecidas, não há grupos a encontrar.

Na prática: é ferramenta para milhares de pontos, não para milhões.

## Exercícios

### Exercício 1: Implementação do Average Linkage

Complete a classe `HierarchicalClustering` adicionando os métodos **Average Linkage** e **Ward Linkage**. Em seguida, teste os quatro critérios no dataset simples (`X_simple`) e compare as distâncias de fusão com as do `linkage` do SciPy.

O average sai direto da lista `pairwise`. O Ward é mais difícil, porque não se calcula a partir das distâncias entre pares: são necessários os centróides dos dois clusters, e para isso `_cluster_distance` precisa receber `X`. Lembre que o SciPy plota $\sqrt{2\Delta}$, e não $\Delta$.

In [ ]:
# Seu código aqui

### Exercício 2: Análise do Dataset Wine

Aplique a clusterização hierárquica do SciPy ao dataset Wine. Padronize os dados antes de calcular qualquer distância, selecione um bom par de features para visualização bidimensional e compare os diferentes métodos de ligação.

In [ ]:
# Seu código aqui

### Exercício 3: Determinação do Número Ótimo de Clusters

Com base no melhor método de ligação identificado no Exercício 2, determine o número ótimo de clusters para o dataset Wine usando análise visual do dendrograma e validação com os rótulos verdadeiros.

In [ ]:
# Seu código aqui